# king - man + woman = ?

## Un viaggio nello spazio degli embeddings

In questo lab esploreremo come i modelli di linguaggio rappresentano le parole nello spazio vettoriale.
Partiremo dalla tokenizzazione, passeremo per l'aritmetica degli embeddings, e arriveremo a toccare
la geometria dello spazio latente - inclusi bias e limiti.

---

In [ ]:
import numpy as np
import tiktoken
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist
import plotly.express as px
import plotly.graph_objects as go

---
## Parte 1: Tokenizzazione

Prima di diventare vettori, il testo deve essere *spezzato in pezzi*. Un tokenizer non divide per parole:
usa un vocabolario appreso (BPE, WordPiece, etc.) che bilancia frequenza e copertura.

Usiamo il tokenizer di GPT-4 (encoding `cl100k_base`).

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")

In [ ]:
print(f"Lungezza del vocabolario:           {enc.n_vocab}")
print(f"Primi tokens                        {[enc.decode([i]) for i in range(10)]}")
print(f"Ultimi tokens                       {[enc.decode([i]) for i in range(enc.n_vocab-30, enc.n_vocab-25)]}")

In [ ]:
text = "Il gatto si siede sul divano"
tokens = enc.encode(text)

print(f"Testo:              {text}")
print(f"Token IDs:          {tokens}")
print(f"Token decodificati: {[enc.decode([t]) for t in tokens]}")
print(f"Numero di token:    {len(tokens)}")

In [ ]:
# Sorprese del tokenizer: lingue, numeri, subword
tests = [
    "The cat sat on the mat",
    "Il gatto sedeva sul tappeto",
    "123", "1234567890",
    "supercalifragilisticespiralidoso",
    "def fibonacci(n):",
    "    ",
]

print(f"{'Testo':<45} {'#tok':>4}  Token")
print("-" * 90)
for t in tests:
    toks = enc.encode(t)
    decoded = [enc.decode([tok]) for tok in toks]
    print(f"{repr(t):<45} {len(toks):>4}  {decoded}")

**Osservazioni:**
- L'italiano usa piu token dell'inglese per lo stesso significato (il vocabolario è sbilanciato)
- I numeri grandi vengono spezzati in modo arbitrario
- Il codice Python ha token dedicati (indentazione!)

---
## Parte 2: Word Embeddings

Carichiamo GloVe (Global Vectors for Word Representation), un modello di word embeddings
pre-addestrato. Ogni parola e un vettore in $\mathbb{R}^{100}$.

GloVe impara le rappresentazioni dalla **co-occorrenza**: parole che appaiono in contesti simili
finiscono vicine nello spazio.

In [ ]:
from gensim.models import KeyedVectors

model: KeyedVectors = KeyedVectors.load('glove-wiki-gigaword-100.kv')
print(f"Vocabolario: {len(model.key_to_index):,} parole")
print(f"Dimensione vettori: {model.vector_size}")

# Ispezioniamo un vettore
word = "king"
vec = model[word]
print(f"\nVettore di '{word}' (prime 10 componenti):")
print(np.round(vec[:10], 4))
print(f"Norma L2: {np.linalg.norm(vec):.4f}")
print(f"Shape: {vec.shape}")

### Similarita e distanza

Due metriche fondamentali:
- **Cosine similarity**: misura l'angolo tra i vettori. Ignora il modulo, cattura la *direzione*.
$$\text{cos\_sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$
- **Distanza euclidea**: misura la distanza geometrica classica.

Per gli embeddings, la cosine similarity è quasi sempre piu informativa. 

In [ ]:
pairs = [
    ("king", "queen"),
    ("king", "car"),
    ("cat", "dog"),
    ("cat", "democracy"),
    ("good", "great"),
    ("good", "bad"),       # semanticamente opposti ma contestualmente vicini!
]

print(f"{'Coppia':<28} {'Cos. Sim':>9} {'Dist. Euc':>10}")
print("-" * 50)
for w1, w2 in pairs:
    cos_sim = model.similarity(w1, w2)
    euc_dist = np.linalg.norm(model[w1] - model[w2])
    print(f"{w1 + ' / ' + w2:<28} {cos_sim:>9.4f} {euc_dist:>10.4f}")

**Nota su good/bad:** hanno alta similarita perche appaiono in contesti simili ("the food was good/bad").
Gli embeddings catturano *distributional similarity*, non significato "logico".
Questo e un limite fondamentale dei modelli distribuzionali.

In [ ]:
# I vicini piu simili di una parola
for target in ["python", "king", "trento"]:
    print(f"\nVicini di '{target}':")
    for word, score in model.most_similar(target, topn=8):
        print(f"  {word:20s} {score:.4f}")


---
### Aritmetica vettoriale

Le **relazioni semantiche** sono codificate come **direzioni** nello spazio.

Se la relazione `man -> woman` e una direzione $\mathbf{d}$, allora:
$$\text{king} - \text{man} + \text{woman} \approx \text{king} + \mathbf{d} \approx \text{queen}$$

Questo funziona perche GloVe/Word2Vec imparano **struttura lineare**.

In [ ]:
analogies = [
    (["king", "woman"], ["man"], "king - man + woman = ?"),
    (["paris", "germany"], ["france"], "paris - france + germany = ?"),
    (["bigger", "cold"], ["big"], "bigger - big + cold = ? (morfologia)"),
    (["walking", "swam"], ["walk"], "walking - walk + swim = ? (tempo verbale)"),
    (["sushi", "italy"], ["japan"], "sushi - japan + italy = ? (cultura)"),
    (["physics", "freud"], ["einstein"], "physics - einstein + freud +  = ? (campo)"),
]

for pos, neg, desc in analogies:
    result = model.most_similar(positive=pos, negative=neg, topn=5)
    print(f"\n{desc}")
    for word, score in result:
        print(f"  {word:20s} {score:.4f}")

---
## Parte 3: Geometria dello spazio

Visualizziamo gli embeddings in uno spazio a bassa dimensionalità per capire come si organizzano.

In [ ]:
# PCA su embeddings raggruppati per categoria semantica
words_by_category = {
    "Royalty":    ["king", "queen", "prince", "princess", "throne", "crown"],
    "Animals":    ["cat", "dog", "horse", "bird", "fish", "lion"],
    "Countries":  ["france", "germany", "italy", "spain", "japan", "brazil"],
    "Capitals":   ["paris", "berlin", "rome", "madrid", "tokyo", "brasilia"],
    "Tech":       ["computer", "software", "algorithm", "database", "network", "code"],
    "Emotions":   ["happy", "sad", "angry", "fear", "love", "hate"],
}

all_words, categories = [], []
for cat, words in words_by_category.items():
    for w in words:
        if w in model.key_to_index:
            all_words.append(w)
            categories.append(cat)

vectors = np.array([model[w] for w in all_words])

pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

fig = px.scatter(
    x=coords[:, 0], y=coords[:, 1],
    text=all_words, color=categories,
    title=f"PCA degli embeddings (varianza spiegata: {pca.explained_variance_ratio_.sum():.1%})",
    labels={"x": f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
            "y": f"PC2 ({pca.explained_variance_ratio_[1]:.1%})"},
    width=850, height=600,
)
fig.update_traces(textposition="top center", marker=dict(size=10))
fig.update_layout(template="plotly_white")
fig.show()


**Osservazioni:** Countries e Capitals formano cluster vicini. 

In [ ]:
# Parallelogramma delle analogie
# Se king:queen = man:woman, le frecce dovrebbero essere nella stessa direzione

pairs = [("man", "woman"), ("king", "queen"), ("uncle", "aunt"), ("boy", "girl"), ("husband", "wife")]
vectors_p = np.array([model[w] for pair in pairs for w in pair])
words_p = [w for pair in pairs for w in pair]

pca_p = PCA(n_components=2)
coords_p = pca_p.fit_transform(vectors_p)

colors = px.colors.qualitative.Set2
fig = go.Figure()
for i, (w1, w2) in enumerate(pairs):
    i1, i2 = i * 2, i * 2 + 1
    fig.add_trace(go.Scatter(
        x=[coords_p[i1, 0], coords_p[i2, 0]],
        y=[coords_p[i1, 1], coords_p[i2, 1]],
        mode="lines+markers+text",
        text=[w1, w2], textposition="top center",
        name=f"{w1} -> {w2}",
        line=dict(color=colors[i % len(colors)], width=2),
        marker=dict(size=10),
    ))
    fig.add_annotation(
        x=coords_p[i2, 0], y=coords_p[i2, 1],
        ax=coords_p[i1, 0], ay=coords_p[i1, 1],
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=2, arrowsize=1.5,
        arrowcolor=colors[i % len(colors)],
    )

fig.update_layout(
    title="Le frecce man->woman sono (quasi) tutter nella stessa direzione",
    width=750, height=500, template="plotly_white",
)
fig.show()


---
### Contextual embeddings

In [ ]:
from transformers import AutoTokenizer, AutoModel

# Caricamento locale
bert_tokenizer = AutoTokenizer.from_pretrained("distilbert-multilingual-cased")
bert_model = AutoModel.from_pretrained("distilbert-multilingual-cased", output_hidden_states=True)


In [ ]:
# Esempio di tokenizzazione (sub-words)
ex_sentence = "Ho mangiato una pesca questa mattina"
tokens = bert_tokenizer(ex_sentence, return_tensors="pt")
print(f"Testo:              {ex_sentence}")
print(f"Token IDs:          {tokens['input_ids'][0].tolist()}")
print(f"Token decodificati: {[bert_tokenizer.decode([t]) for t in tokens['input_ids'][0].tolist()]}")


In [ ]:
import torch
contextual_sentences = {
    "attività": [
        "La pesca in mare è vietata in questa zona protetta.",
        "Mio zio pratica la pesca sportiva ogni domenica mattina.",
        "La pesca al lago richiede molta pazienza.",
        "Durante la pesca notturna hanno usato lampade molto potenti.",
        "La pesca con la rete è regolata da norme severe.",
        "Domani andremo a pesca sul fiume con alcuni amici.",
        "La stagione della pesca del tonno inizia a maggio.",
        "La pesca subacquea è vietata vicino al porto.",
        "Per la pesca alla trota serve un permesso speciale.",
        "La pesca commerciale ha un forte impatto sugli ecosistemi marini.",
    ],
    "frutto": [
        "Ho mangiato una pesca molto dolce dopo pranzo.",
        "Al mercato ho comprato una pesca matura e profumata.",
        "La pesca che ho tagliato era succosa e gialla.",
        "Per dessert abbiamo servito yogurt con pesca fresca.",
        "Questa pesca è troppo acerba per essere mangiata oggi.",
        "Il sapore della pesca mi ricorda l'estate.",
        "Ho preparato una torta con fettine di pesca.",
        "La marmellata di pesca fatta in casa era ottima.",
        "Nel cestino c'era una pesca morbida e molto profumata.",
        "Il gelato alla pesca era fresco e leggero.",
    ]
}
bert_model.eval()  # disabilita dropout
target = "pesca"

vectors = []
categories = []
texts = []
for category, sentences in contextual_sentences.items():
    for sent in sentences:
        tokenized_sentence = bert_tokenizer(sent, return_tensors="pt")
        # Trova l'indice della parola target nella frase tokenizzata
        try:
            token_index = np.nonzero(tokenized_sentence.input_ids[0] == bert_tokenizer.encode(target, add_special_tokens=False)[0]).item()
        except:
            print(sent)
            continue
        if token_index is not None:
            with torch.no_grad():
                outputs = bert_model(**tokenized_sentence)
                # Prendi il last hidden state del token target come embedding contestuale
                # Output del modello (prima della head): (batch_size, seq_len, hidden_dim)
                target_embedding = outputs.last_hidden_state[0, token_index, :].numpy()
                vectors.append(target_embedding)
                categories.append(category)
                texts.append(sent)
        
pca = PCA(n_components=3)
coords = pca.fit_transform(vectors)

fig = px.scatter_3d(
    x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
    color=categories, hover_name=texts,
    title=f"PCA degli embeddings (varianza spiegata: {pca.explained_variance_ratio_.sum():.1%})",
    labels={"x": f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
            "y": f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
            "z": f"PC3 ({pca.explained_variance_ratio_[2]:.1%})"},
    width=900, height=700,
)
fig.update_traces(marker=dict(size=10))
fig.update_layout(template="plotly_white")
fig.show()


---
### Bias negli embeddings

Se la direzione `man -> woman` codifica il genere, possiamo proiettare qualsiasi parola su questa direzione.
Cosa succede con le professioni?

In [ ]:
# Direzione di genere
gender_dir = model["woman"] - model["man"]
gender_dir_norm = gender_dir / np.linalg.norm(gender_dir)

professions = [
    "doctor", "nurse", "engineer", "teacher", "programmer",
    "secretary", "scientist", "receptionist", "pilot", "librarian",
    "surgeon", "dancer", "mathematician", "hairdresser", "professor",
    "nanny", "lawyer", "maid", "architect", "chef",
]

projections = [(p, float(np.dot(model[p], gender_dir_norm))) for p in professions]
projections.sort(key=lambda x: x[1])

fig = go.Figure()
fig.add_trace(go.Bar(
    y=[p[0] for p in projections],
    x=[p[1] for p in projections],
    orientation="h",
    marker_color=["#e74c3c" if p[1] < 0 else "#3498db" for p in projections],
))
fig.update_layout(
    title="Proiezione delle professioni sulla direzione man -> woman",
    xaxis_title="<-- man                    woman -->",
    height=550, width=700, template="plotly_white",
)
fig.show()

- Questi bias riflettono i testi di addestramento (Wikipedia, news)
- Un LLM che usa questi embeddings *perpetua e amplifica* i bias del corpus
- Debiasing e un problema aperto

---
### La maledizione della dimensionalita

In alta dimensione, l'intuizione geometrica collassa. 

In [ ]:
# In alta dimensione, i punti random diventano quasi equidistanti
dims = [2, 10, 50, 100, 300, 1000]
n_points = 500

print(f"{'Dim':>6} | {'Media dist':>10} | {'Std dist':>10} | {'CV':>8}")
print("-" * 45)
for d in dims:
    points = np.random.randn(n_points, d)
    dists = pdist(points)
    cv = dists.std() / dists.mean()
    print(f"{d:>6} | {dists.mean():>10.2f} | {dists.std():>10.4f} | {cv:>8.4f}")

print("\nIl coefficiente di variazione (CV) tende a 0")


---
## Sfide

Esercizi con gli embeddings.


### Sfida: Analogia

Riesci a trovare positive e negative per ottenere **"submarine"**?

Hai a disposizione 2 parole positive e 1 negativa. 


In [ ]:
# ============================================================
# SFIDA: Trova positive e negative per ottenere "submarine"
# ============================================================
# Usa: model.most_similar(positive=[...], negative=[...], topn=5)

positive = ["...", "..."]   # <-- MODIFICA
negative = ["..."]          # <-- MODIFICA

risultato = model.most_similar(positive=positive, negative=negative, topn=5)
for parola, punteggio in risultato:
    print(f"  {parola:20s} {punteggio:.4f}")


### Sfida: Trova l'intruso

Data una lista di parole, scrivi una funzione che trovi quella che non appartiene al gruppo.


In [ ]:
def trova_intruso(model, parole):
    """Restituisce la parola che non appartiene al gruppo."""
    # Il tuo codice qui
    pass

# Testa la tua funzione:
# trova_intruso(model, ["cat", "dog", "horse", "computer", "fish"])

### Sfida: Cluster compatto

Scegli 5 parole del vocabolario GloVe che massimizzino la **similarita media** tra tutte le coppie.

In [ ]:
# ============================================================
# SFIDA: Trova il cluster piu compatto
# ============================================================

parole = ["...", "...", "...", "...", "..."]  # <-- SCEGLI LE TUE 5 PAROLE

# Calcola la similarita media tra tutte le coppie
n = len(parole)
similarita = []
for i in range(n):
    for j in range(i + 1, n):
        sim = model.similarity(parole[i], parole[j])
        similarita.append(sim)

media = np.mean(similarita)
print(f"Parole: {parole}")
print(f"Similarita media: {media:.4f}")


### Riflessione: PCA e angoli

Abbiamo visualizzato le direzioni (es. man→woman) nello spazio PCA a 2 componenti.
Ma la PCA non preserva gli angoli tra vettori in generale.

**Domanda:** come possiamo verificare che due direzioni sono davvero parallele
nello spazio originale a 100 dimensioni, senza fidarci del plot? Se lo trovi, prova ad applicarlo al caso di `man->woman` e `king->queen`.